# Closed-World Re-ID Evaluation (Video-to-Video & Image-to-Video)

This notebook evaluates distance matrices generated by models in the **Closed-World** setting.
It computes and prints:
- **mAP** (mean Average Precision) with 95% Bootstrap Confidence Intervals
- **Rank-1** Cumulative Matching Characteristic with 95% Bootstrap Confidence Intervals
- **Rank-5** Cumulative Matching Characteristic with 95% Bootstrap Confidence Intervals


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add root directory to sys.path
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR if (CURRENT_DIR / "evaluation").exists() else CURRENT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
if str(ROOT_DIR / "evaluation") not in sys.path:
    sys.path.append(str(ROOT_DIR / "evaluation"))

from evaluation.evaluation_utils import bootstrap_from_csv
print("Loaded evaluation_utils successfully!")
print(f"Project root: {ROOT_DIR}")

Loaded evaluation_utils successfully!
Project root: ./


In [ ]:
# Configuration: Models and Candidate CSV Paths
MODELS = ["bot", "agw", "arbase", "sbs", "mgn", "tfclip", "clip", "baseline", "megadescriptor", "scl", "adafreq", "rovf"]
MODALITIES = ["vid2vid", "img2vid"]

SEARCH_DIRS = [
    ROOT_DIR / "evaluation" / "csvs_closed_results",
]

def find_closed_csv(model_name: str, modality: str):
    """Find closed-world distance matrix CSV for a given model and modality."""
    names = [model_name]
    if model_name == "baseline":
        names.extend(["dinov2", "dinov2_baseline", "dinov2_bot"])
    elif model_name in ("clip", "clip_baseline"):
        names.extend(["clip_baseline", "clip_backbone", "clip"])
    elif model_name == "megadescriptor":
        names.extend(["megadescriptor", "closed_megadescriptor", "md"])
    elif model_name == "scl":
        names.extend(["scl_attn", "scl"])
    elif model_name == "adafreq":
        names.extend(["adafreq_attn", "adafreq"])
    elif model_name == "rovf":
        names.extend(["rovf_s_ep10", "rovf_s_ep5", "rovf_base_ep10", "rovf_base_ep5", "rovf"])

    for name in names:
        patterns = [
            f"{name}_{modality}/closed_dist_matrix.csv",
            f"{name}_{modality}/vid2vid_closed_dist_matrix.csv",
            f"{name}_{modality}/img2vid_closed_dist_matrix.csv",
            f"{name}_closed/closed_dist_matrix.csv",
            f"{name}_closed_{modality}/closed_dist_matrix.csv",
        ]
        for search_dir in SEARCH_DIRS:
            if not search_dir.exists():
                continue
            for pat in patterns:
                candidate = search_dir / pat
                if candidate.exists() and candidate.stat().st_size > 1000:
                    return candidate
            folder = search_dir / f"{name}_{modality}"
            if folder.exists():
                for f in folder.glob("*closed*.csv"):
                    if f.stat().st_size > 1000:
                        return f
    return None

models_to_test = {}
for m in MODELS:
    for mod in MODALITIES:
        if m in ("tfclip", "rovf") and mod == "img2vid":
            continue  # TF-CLIP and RoVF are strictly video-to-video methods
        key = f"{m.upper()} ({mod})"
        csv_path = find_closed_csv(m, mod)
        if csv_path:
            models_to_test[key] = str(csv_path)
            print(f"[FOUND] {key:<24} -> {csv_path.relative_to(ROOT_DIR)}")
        else:
            print(f"[PENDING] {key:<22} -> Training or evaluation in progress...")


[FOUND] BOT (vid2vid)            -> evaluation/csvs_closed_results/bot_vid2vid/closed_dist_matrix.csv
[FOUND] BOT (img2vid)            -> evaluation/csvs_closed_results/bot_img2vid/closed_dist_matrix.csv
[FOUND] AGW (vid2vid)            -> evaluation/csvs_closed_results/agw_vid2vid/closed_dist_matrix.csv
[FOUND] AGW (img2vid)            -> evaluation/csvs_closed_results/agw_img2vid/closed_dist_matrix.csv
[FOUND] ARBASE (vid2vid)         -> evaluation/csvs_closed_results/arbase_vid2vid/closed_dist_matrix.csv
[FOUND] ARBASE (img2vid)         -> evaluation/csvs_closed_results/arbase_img2vid/closed_dist_matrix.csv
[FOUND] SBS (vid2vid)            -> evaluation/csvs_closed_results/sbs_vid2vid/closed_dist_matrix.csv
[FOUND] SBS (img2vid)            -> evaluation/csvs_closed_results/sbs_img2vid/closed_dist_matrix.csv
[FOUND] MGN (vid2vid)            -> evaluation/csvs_closed_results/mgn_vid2vid/closed_dist_matrix.csv
[FOUND] MGN (img2vid)            -> evaluation/csvs_closed_results/mgn_img2v

In [ ]:
# Run Bootstrap Evaluation
M_ITERATIONS = 100
all_closed_results = {}

for name, path in models_to_test.items():
    print("=" * 60)
    print(f"Evaluating {name} on {Path(path).name} ...")
    print("=" * 60)
    try:
        res = bootstrap_from_csv(path, m=M_ITERATIONS, mode="closed")
        all_closed_results[name] = res
    except Exception as e:
        print(f"[ERROR] Failed to evaluate {name}: {e}")


Evaluating BOT (vid2vid) on closed_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
Evaluating BOT (img2vid) on closed_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
Evaluating AGW (vid2vid) on closed_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
Evaluating AGW (img2vid) on closed_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
Evaluating ARBASE (vid2vid) on closed_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
Evaluating ARBASE (img2vid) on closed_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed closed metrics for c

In [ ]:
# Display Closed-World Benchmark Table
rows = []
for m in MODELS:
    for mod in MODALITIES:
        if m in ("tfclip", "rovf") and mod == "img2vid":
            continue  # TF-CLIP and RoVF are strictly video-to-video methods
        name = f"{m.upper()} ({mod})"
        if name in all_closed_results:
            res = all_closed_results[name]
            full = res.get("full_set", {})
            
            # mAP
            m_mean = res.get("mAP_boot_mean", full.get("mAP", 0.0)) * 100
            m_std = res.get("mAP_std", 0.0) * 100
            m_ci = res.get("mAP_ci", (0.0, 0.0))
            m_low, m_high = m_ci[0] * 100, m_ci[1] * 100
            
            # CMC Rank-1, Rank-5
            cmc_boot = res.get("cmc_boot_mean", full.get("cmc", np.zeros(10)))
            cmc_std = res.get("cmc_std", np.zeros(10))
            cmc_l = res.get("cmc_ci_lower", cmc_boot)
            cmc_u = res.get("cmc_ci_upper", cmc_boot)
            
            r1 = cmc_boot[0] * 100
            r1_std = cmc_std[0] * 100 if len(cmc_std) > 0 else 0.0
            r1_l, r1_u = cmc_l[0] * 100, cmc_u[0] * 100
            
            r5 = cmc_boot[4] * 100 if len(cmc_boot) >= 5 else 0.0
            r5_std = cmc_std[4] * 100 if len(cmc_std) >= 5 else 0.0
            r5_l = cmc_l[4] * 100 if len(cmc_l) >= 5 else 0.0
            r5_u = cmc_u[4] * 100 if len(cmc_u) >= 5 else 0.0
            
            rows.append({
                "Model (Modality)": name,
                "mAP (%)": f"{m_mean:.1f} ± {m_std:.1f} [{m_low:.1f}-{m_high:.1f}]",
                "Rank-1 (%)": f"{r1:.1f} ± {r1_std:.1f} [{r1_l:.1f}-{r1_u:.1f}]",
                "Rank-5 (%)": f"{r5:.1f} ± {r5_std:.1f} [{r5_l:.1f}-{r5_u:.1f}]",
                "Status": "Verified"
            })
        else:
            rows.append({
                "Model (Modality)": name,
                "mAP (%)": "Pending...",
                "Rank-1 (%)": "Pending...",
                "Rank-5 (%)": "Pending...",
                "Status": "In Progress"
            })

df_closed = pd.DataFrame(rows)

print("=" * 115)
print(f"{'Model (Modality)':<22} | {'mAP [95% CI]':<26} | {'Rank-1 [95% CI]':<26} | {'Rank-5 [95% CI]':<26} | {'Status':<10}")
print("=" * 115)
for r in rows:
    print(f"{r['Model (Modality)']:<22} | {r['mAP (%)']:<26} | {r['Rank-1 (%)']:<26} | {r['Rank-5 (%)']:<26} | {r['Status']:<10}")
print("-" * 115)

display(df_closed)


Model (Modality)       | mAP [95% CI]               | Rank-1 [95% CI]            | Rank-5 [95% CI]            | Status    
BOT (vid2vid)          | 24.6 ± 1.0 [22.6-26.8]     | 24.3 ± 1.3 [21.4-26.7]     | 45.6 ± 1.7 [42.5-48.8]     | Verified  
BOT (img2vid)          | 18.7 ± 0.9 [17.1-20.3]     | 17.5 ± 1.1 [15.4-19.6]     | 35.7 ± 1.5 [32.8-38.2]     | Verified  
AGW (vid2vid)          | 27.0 ± 1.1 [25.0-29.5]     | 26.6 ± 1.4 [24.5-29.7]     | 49.1 ± 1.6 [46.2-52.7]     | Verified  
AGW (img2vid)          | 21.5 ± 0.9 [19.5-23.5]     | 21.6 ± 1.1 [19.3-23.4]     | 40.0 ± 1.4 [36.8-42.9]     | Verified  
ARBASE (vid2vid)       | 31.3 ± 1.1 [29.4-33.4]     | 32.7 ± 1.4 [30.2-35.5]     | 55.5 ± 1.7 [51.9-58.5]     | Verified  
ARBASE (img2vid)       | 23.7 ± 0.9 [21.7-25.4]     | 23.0 ± 1.2 [20.2-24.8]     | 43.3 ± 1.4 [40.5-45.8]     | Verified  
SBS (vid2vid)          | 30.9 ± 1.1 [29.1-33.2]     | 31.4 ± 1.4 [28.9-34.4]     | 54.8 ± 1.6 [51.6-58.3]     | Verified  
SBS (img2vid)   

,Model (Modality),mAP (%),Rank-1 (%),Rank-5 (%),Status
0,BOT (vid2vid),24.6 ± 1.0 [22.6-26.8],24.3 ± 1.3 [21.4-26.7],45.6 ± 1.7 [42.5-48.8],Verified
1,BOT (img2vid),18.7 ± 0.9 [17.1-20.3],17.5 ± 1.1 [15.4-19.6],35.7 ± 1.5 [32.8-38.2],Verified
2,AGW (vid2vid),27.0 ± 1.1 [25.0-29.5],26.6 ± 1.4 [24.5-29.7],49.1 ± 1.6 [46.2-52.7],Verified
3,AGW (img2vid),21.5 ± 0.9 [19.5-23.5],21.6 ± 1.1 [19.3-23.4],40.0 ± 1.4 [36.8-42.9],Verified
4,ARBASE (vid2vid),31.3 ± 1.1 [29.4-33.4],32.7 ± 1.4 [30.2-35.5],55.5 ± 1.7 [51.9-58.5],Verified
5,ARBASE (img2vid),23.7 ± 0.9 [21.7-25.4],23.0 ± 1.2 [20.2-24.8],43.3 ± 1.4 [40.5-45.8],Verified
6,SBS (vid2vid),30.9 ± 1.1 [29.1-33.2],31.4 ± 1.4 [28.9-34.4],54.8 ± 1.6 [51.6-58.3],Verified
7,SBS (img2vid),23.9 ± 0.9 [22.2-25.9],23.3 ± 1.1 [21.3-25.2],43.8 ± 1.5 [41.0-46.7],Verified
8,MGN (vid2vid),26.3 ± 1.1 [24.3-28.3],26.9 ± 1.3 [24.6-29.7],47.2 ± 1.7 [44.0-50.3],Verified
9,MGN (img2vid),20.4 ± 0.9 [18.8-22.0],19.8 ± 1.2 [17.7-21.6],38.2 ± 1.5 [35.6-40.5],Verified
